In [ ]:
from ultralytics import YOLO

# Training
import torch
print(torch.__version__)
print(torch.version.cuda)

2.10.0+cu128
12.8


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
import zipfile
import os

zip_path = '/content/Bridge crack dataset.v10i.yolov8-obb.zip'
extract_to = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Extracted successfully!")
print(os.listdir(extract_to))

Extracted successfully!
['train', 'README.roboflow.txt', 'data.yaml', 'test', 'valid', 'README.dataset.txt']


In [ ]:


model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=300,
    imgsz=640,
    plots=True,
    device=0,
    augment=True,       # Enable built-in augmentations
    fliplr=0.5,
    degrees=15.0,       # Rotation helps with angled cracks
    patience=30         # Early stopping
)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, pers

In [ ]:
import os
import glob
import cv2
import math
from ultralytics import YOLO
from fpdf import FPDF

# ==========================================
# MODULE 1: AI INFERENCE & SEVERITY SCORING
# ==========================================
def analyze_infrastructure_batch(image_folder, model_path):
    print(f"\n[SYSTEM] Booting The Inspector's Eye...")
    print(f"[SYSTEM] Loading custom weights: {model_path}")
    model = YOLO(model_path)

    # Cross-platform image aggregation
    all_images = []
    valid_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']

    for ext in valid_extensions:
        search_pattern = os.path.join(image_folder, ext)
        all_images.extend(glob.glob(search_pattern))

    if not all_images:
        print(f"[ERROR] No valid image files found in '{image_folder}'.")
        return [], ""

    print(f"[SYSTEM] Found {len(all_images)} images. Initiating batch inference...")

    # Run bulk inference
    results = model.predict(source=all_images, conf=0.25, save=True)

    all_report_data = []
    save_directory = results[0].save_dir if results else ""

    print("[SYSTEM] Processing geometry and severity scoring...")

    # Pair original paths with YOLO results to prevent FileNotFoundError
    for orig_path, r in zip(all_images, results):

        # Read true image dimensions
        img = cv2.imread(orig_path)
        img_height, img_width = img.shape[:2]
        image_diagonal = math.hypot(img_width, img_height)

        # Path management
        yolo_filename = os.path.basename(r.path)
        annotated_image_path = os.path.join(r.save_dir, yolo_filename)
        real_filename = os.path.basename(orig_path)

        image_data = {
            "filename": real_filename,
            "annotated_path": annotated_image_path,
            "defects": []
        }

        for i, box in enumerate(r.boxes):
            coords = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            x1, y1, x2, y2 = coords

            # Civil Engineering Heuristic: Diagonal Crack Length
            box_w = x2 - x1
            box_h = y2 - y1
            crack_length = math.hypot(box_w, box_h)

            ratio = crack_length / image_diagonal

            # Severity Classification
            if ratio > 0.40:
                severity = "CRITICAL"
            elif ratio > 0.15:
                severity = "MAJOR"
            else:
                severity = "MINOR"

            image_data["defects"].append({
                "id": i + 1,
                "confidence": round(conf * 100, 2),
                "severity": severity,
                "ratio": round(ratio * 100, 2) # Stored as a percentage
            })

        all_report_data.append(image_data)

    return all_report_data, save_directory

# ==========================================
# MODULE 2: AUTOMATED PDF GENERATION
# ==========================================
def generate_pdf_report(report_data, save_dir):
    if not report_data:
        return

    print("[SYSTEM] Compiling Final PDF Report...")
    pdf = FPDF()

    # --- COVER PAGE ---
    pdf.add_page()
    pdf.set_font("Arial", 'B', 24)
    pdf.ln(20)
    pdf.cell(0, 20, txt="The Inspector's Eye", ln=True, align='C')
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="Automated Structural Analysis Report", ln=True, align='C')
    pdf.ln(20)

    # Global Statistics
    total_images = len(report_data)
    total_cracks = sum(len(img["defects"]) for img in report_data)
    total_critical = sum(sum(1 for d in img["defects"] if d["severity"] == "CRITICAL") for img in report_data)

    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, txt=f"Total Infrastructure Nodes Analyzed: {total_images}", ln=True)
    pdf.cell(0, 10, txt=f"Total Structural Defects Detected: {total_cracks}", ln=True)

    pdf.set_font("Arial", 'B', 12)
    pdf.set_text_color(255, 0, 0) if total_critical > 0 else pdf.set_text_color(0, 128, 0)
    pdf.cell(0, 10, txt=f"Immediate Action Required (Critical Faults): {total_critical}", ln=True)
    pdf.set_text_color(0, 0, 0)

    # --- DETAILED ANALYSIS PAGES ---
    for img_data in report_data:
        pdf.add_page()

        pdf.set_font("Arial", 'B', 14)
        pdf.cell(0, 10, txt=f"Node Analysis: {img_data['filename']}", ln=True)
        pdf.ln(5)

        if os.path.exists(img_data['annotated_path']):
            pdf.image(img_data['annotated_path'], w=170)
        else:
            pdf.set_font("Arial", 'I', 10)
            pdf.cell(0, 10, txt="[Vision system failed to render annotated output]", ln=True)

        pdf.ln(10)

        if not img_data["defects"]:
            pdf.set_font("Arial", 'B', 12)
            pdf.set_text_color(0, 128, 0)
            pdf.cell(0, 10, txt="STATUS: NO STRUCTURAL DEFECTS DETECTED.", ln=True)
            pdf.set_text_color(0, 0, 0)
        else:
            pdf.set_font("Arial", 'B', 10)
            pdf.cell(30, 10, "Defect ID", border=1, align='C')
            pdf.cell(50, 10, "Severity", border=1, align='C')
            pdf.cell(50, 10, "Span Ratio (%)", border=1, align='C')
            pdf.cell(50, 10, "AI Confidence (%)", border=1, align='C')
            pdf.ln()

            pdf.set_font("Arial", '', 10)
            for d in img_data["defects"]:
                pdf.cell(30, 10, str(d["id"]), border=1, align='C')
                pdf.cell(50, 10, d["severity"], border=1, align='C')
                pdf.cell(50, 10, str(d["ratio"]), border=1, align='C')
                pdf.cell(50, 10, str(d["confidence"]), border=1, align='C')
                pdf.ln()

    # --- FINAL EXPORT ---
    output_pdf_path = os.path.join(save_dir, "Final_Inspection_Report.pdf")
    pdf.output(output_pdf_path)
    print(f"\n==========================================")
    print(f"[SUCCESS] Report saved to: {output_pdf_path}")
    print(f"==========================================")


MODEL_WEIGHTS = 'runs/detect/train5/weights/best.pt'
TARGET_DIRECTORY = '/content/dataset/test/images'

# 2. Execute the Pipeline
extracted_data, output_directory = analyze_infrastructure_batch(TARGET_DIRECTORY, MODEL_WEIGHTS)

# 3. Generate the Report
if extracted_data:
    generate_pdf_report(extracted_data, output_directory)


[SYSTEM] Booting The Inspector's Eye...
[SYSTEM] Loading custom weights: runs/detect/train5/weights/best.pt
[SYSTEM] Found 18 images. Initiating batch inference...

0: 640x640 1 crack, 26.9ms
1: 640x640 3 cracks, 26.9ms
2: 640x640 6 cracks, 26.9ms
3: 640x640 (no detections), 26.9ms
4: 640x640 (no detections), 26.9ms
5: 640x640 (no detections), 26.9ms
6: 640x640 2 cracks, 26.9ms
7: 640x640 1 crack, 26.9ms
8: 640x640 (no detections), 26.9ms
9: 640x640 (no detections), 26.9ms
10: 640x640 3 cracks, 26.9ms
11: 640x640 2 cracks, 26.9ms
12: 640x640 (no detections), 26.9ms
13: 640x640 2 cracks, 26.9ms
14: 640x640 2 cracks, 26.9ms
15: 640x640 2 cracks, 26.9ms
16: 640x640 (no detections), 26.9ms
17: 640x640 3 cracks, 26.9ms
Speed: 3.7ms preprocess, 26.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict
[SYSTEM] Processing geometry and severity scoring...
[SYSTEM] Compiling Final PDF Report...

[SUCCESS] Report saved to: /content/run

In [ ]:
import shutil
from google.colab import files

# Zip the entire run folder
shutil.make_archive('/content/crack_detection_results', 'zip', '/content/runs/detect')

# Download the zip
files.download('/content/crack_detection_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>